# Lesson 4h — Scaling up to a *real* GPT (the GPT-3 recipe), on your Mac

The **runnable companion** to the *"from 25 letters to GPT-3"* webpage, and the capstone of Lesson 4.

In **4g** we built a *character* model that could spell any word from ~25 letters. That was the leap to "general purpose". This notebook makes the next leap: the **same machine, the grown-up recipe** — the exact architecture as **GPT-3**, just small enough to train on one Mac.

> **GPT-3 is a *family*, not one model.** The GPT-3 paper (Table 2.1) defines **8 sizes from 125M to 175B** parameters, *all with this identical recipe*. The 175B one needed thousands of GPUs. What we build here is the **baby of the same family** — a true GPT-3 architecture, sized for a laptop.

| Model | Params | Layers | d_model | Heads | Context |
|---|---|---|---|---|---|
| **GPT-3 Small** | **125M** | 12 | 768 | 12 | 2048 |
| GPT-3 Medium | 350M | 24 | 1024 | 16 | 2048 |
| GPT-3 XL | 1.3B | 24 | 2048 | 24 | 2048 |
| … | … | … | … | … | … |
| GPT-3 175B | 175B | 96 | 12288 | 96 | 2048 |

We'll train something around **5–10M params** here so it runs in minutes; the `course/minigpt/` folder has the full version for the real overnight run.

In [1]:
import math, os, urllib.request
from dataclasses import dataclass
import torch, torch.nn as nn
from torch.nn import functional as F
import tiktoken

torch.manual_seed(1337)

# Apple Silicon GPU = "mps" (Metal Performance Shaders). It's the Mac equivalent
# of CUDA on an NVIDIA card. Falls back to the CPU if unavailable.
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

device: mps


## 1. The token change: characters → **subword** pieces (BPE)

4g used **single characters** (~25 ids). Real GPTs use a middle ground called **BPE** (byte-pair encoding): common words stay whole, rare words split into chunks, right down to letters if needed. Best of both — short to read, but still able to spell *anything*.

We borrow GPT-2's exact tokenizer via `tiktoken`: a fixed vocabulary of **50,257** subword tokens. No training needed — it's the same one OpenAI used.

In [2]:
enc = tiktoken.get_encoding("gpt2")   # GPT-2 / GPT-3 BPE, 50257 tokens
print("vocab size:", enc.n_vocab)

for s in ["the cat sat", "dinosaur", "antidisestablishmentarianism"]:
    ids = enc.encode(s)
    pieces = [enc.decode([i]) for i in ids]
    print(f"{s!r:35s} -> {ids}  pieces={pieces}")

vocab size: 50257
'the cat sat'                       -> [1169, 3797, 3332]  pieces=['the', ' cat', ' sat']
'dinosaur'                          -> [67, 21317]  pieces=['d', 'inosaur']
'antidisestablishmentarianism'      -> [415, 29207, 44390, 3699, 1042]  pieces=['ant', 'idis', 'establishment', 'arian', 'ism']


Notice `dinosaur` becomes a couple of chunks and the long silly word becomes several — but every string is representable. That open vocabulary is what lets one model read anything on the internet.

## 2. The model — same pipeline as 4g, with the grown-up recipe

It's still **embed → attention → feed-forward → head → softmax** with a causal mask. The upgrades that turn the 4g toy into the GPT-3 architecture:

| Piece | 4g CharGPT | here (GPT-3 recipe) |
|---|---|---|
| token | single character | **BPE subword** |
| norm | (encoder default) | **pre-LayerNorm** (LN *before* attention & MLP) |
| activation | ReLU | **GELU** |
| output head | separate `Linear` | **weight-tied** to the embedding table |
| extras | — | dropout, scaled residual init |

First, the **config** — every "dial" in one place. This *is* the GPT-3 family table as code.

In [3]:
@dataclass
class GPTConfig:
    block_size: int = 128      # context length (tokens seen at once)
    vocab_size: int = 50257    # tiktoken gpt2
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 192          # d_model — width of every vector
    dropout: float = 0.1
    bias: bool = True

**Causal self-attention** — every token looks only at tokens to its *left*. We use PyTorch's fast `scaled_dot_product_attention` with `is_causal=True`, which bakes in the same triangular mask you built by hand in 4g.

In [4]:
class CausalSelfAttention(nn.Module):
    def __init__(self, c):
        super().__init__()
        assert c.n_embd % c.n_head == 0
        self.c_attn = nn.Linear(c.n_embd, 3 * c.n_embd, bias=c.bias)  # q,k,v at once
        self.c_proj = nn.Linear(c.n_embd, c.n_embd, bias=c.bias)
        self.attn_drop = nn.Dropout(c.dropout)
        self.resid_drop = nn.Dropout(c.dropout)
        self.n_head, self.n_embd, self.dropout = c.n_head, c.n_embd, c.dropout

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.c_proj(y))

The **feed-forward** (each token thinks on its own, 4× wider in the middle, GELU) and the **block** that wires attention + FFN with **pre-LayerNorm** and residual `x = x + sublayer(ln(x))`.

In [5]:
class MLP(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c_fc = nn.Linear(c.n_embd, 4 * c.n_embd, bias=c.bias)
        self.c_proj = nn.Linear(4 * c.n_embd, c.n_embd, bias=c.bias)
        self.drop = nn.Dropout(c.dropout)
    def forward(self, x):
        return self.drop(self.c_proj(F.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ln_1 = nn.LayerNorm(c.n_embd, bias=c.bias); self.attn = CausalSelfAttention(c)
        self.ln_2 = nn.LayerNorm(c.n_embd, bias=c.bias); self.mlp = MLP(c)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))   # look at neighbours
        x = x + self.mlp(self.ln_2(x))    # think alone
        return x

The full **GPT**: token + position embeddings → a stack of blocks → final LayerNorm → head. The head shares weights with the token embedding (**weight tying**) — fewer params, better results.

In [6]:
class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.cfg = c
        self.tf = nn.ModuleDict(dict(
            wte=nn.Embedding(c.vocab_size, c.n_embd),   # token embeddings
            wpe=nn.Embedding(c.block_size, c.n_embd),   # position embeddings
            drop=nn.Dropout(c.dropout),
            h=nn.ModuleList(Block(c) for _ in range(c.n_layer)),
            ln_f=nn.LayerNorm(c.n_embd, bias=c.bias),
        ))
        self.lm_head = nn.Linear(c.n_embd, c.vocab_size, bias=False)
        self.tf.wte.weight = self.lm_head.weight                 # weight tying
        self.apply(self._init)
        for n, p in self.named_parameters():                     # scaled residual init
            if n.endswith("c_proj.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * c.n_layer))

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tf.drop(self.tf.wte(idx) + self.tf.wpe(pos))
        for blk in self.tf.h: x = blk(x)
        x = self.tf.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            return logits, loss
        return self.lm_head(x[:, [-1], :]), None

    @torch.no_grad()
    def generate(self, idx, n, temperature=0.8, top_k=200):
        for _ in range(n):
            logits, _ = self(idx[:, -self.cfg.block_size:])
            logits = logits[:, -1, :] / temperature
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            idx = torch.cat((idx, torch.multinomial(probs, 1)), dim=1)
        return idx

In [7]:
cfg = GPTConfig()
model = GPT(cfg).to(device)
n_params = sum(p.numel() for p in model.parameters()) - model.tf.wpe.weight.numel()
print(f"model: {cfg.n_layer} layers, d_model={cfg.n_embd}, {cfg.n_head} heads")
print(f"parameters: {n_params/1e6:.1f}M  (GPT-3 175B is ~{175000/ (n_params/1e6):.0f}x bigger)")

model: 4 layers, d_model=192, 4 heads
parameters: 11.4M  (GPT-3 175B is ~15312x bigger)


## 3. Data — one long stream of BPE tokens

A character model wants one long string; this model wants one long stream of **token ids**. We use Tiny Shakespeare (~1 MB) so it downloads instantly. (The `minigpt` folder uses **TinyStories** for the real run.)

First, let's actually **look at the corpus** — the raw text, then the same text as the tokens the model will see.


In [8]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = urllib.request.urlopen(url).read().decode("utf-8")

print("=== raw corpus, first 280 characters ===")
print(text[:280])

ids = torch.tensor(enc.encode_ordinary(text), dtype=torch.long)
print("\n=== same opening as BPE tokens ===")
head = ids[:12].tolist()
print("token ids:", head)
print("pieces:   ", [enc.decode([i]) for i in head])

n = int(0.9 * len(ids))
train_data, val_data = ids[:n], ids[n:]
print(f"\n{len(text):,} chars -> {len(ids):,} BPE tokens   (train {len(train_data):,} / val {len(val_data):,})")


=== raw corpus, first 280 characters ===
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

F

=== same opening as BPE tokens ===
token ids: [5962, 22307, 25, 198, 8421, 356, 5120, 597, 2252, 11, 3285, 502]
pieces:    ['First', ' Citizen', ':', '\n', 'Before', ' we', ' proceed', ' any', ' further', ',', ' hear', ' me']

1,115,394 chars -> 338,025 BPE tokens   (train 304,222 / val 33,803)


## 4. The training recipe — the same 5 lines, dressed up

The loop is still **predict → loss → backward → step → repeat** from Lesson 1. The grown-up extras: **AdamW** with weight decay, a **cosine learning-rate schedule with warmup**, and **gradient clipping**. We slide a window to make (input, next-token) batches — exactly like 4g, just with subword ids.

In [9]:
block, batch = cfg.block_size, 16
def get_batch(d):
    ix = torch.randint(len(d) - block - 1, (batch,))
    x = torch.stack([d[i:i+block] for i in ix])
    y = torch.stack([d[i+1:i+1+block] for i in ix])
    return x.to(device), y.to(device)

max_iters, warmup, lr, min_lr = 600, 60, 6e-4, 6e-5
def lr_at(it):
    if it < warmup: return lr * (it + 1) / warmup
    r = (it - warmup) / (max_iters - warmup)
    return min_lr + 0.5 * (1 + math.cos(math.pi * r)) * (lr - min_lr)

# AdamW with weight decay on 2D weights only
decay = [p for p in model.parameters() if p.dim() >= 2]
nodecay = [p for p in model.parameters() if p.dim() < 2]
opt = torch.optim.AdamW([{"params": decay, "weight_decay": 0.1},
                         {"params": nodecay, "weight_decay": 0.0}], lr=lr, betas=(0.9, 0.95))

@torch.no_grad()
def est_val():
    model.eval()
    L = torch.stack([model(*get_batch(val_data))[1] for _ in range(20)]).mean().item()
    model.train(); return L

In [10]:
for it in range(max_iters + 1):
    for g in opt.param_groups: g["lr"] = lr_at(it)
    if it % 150 == 0:
        print(f"iter {it:>4} | val loss {est_val():.3f} | lr {opt.param_groups[0]['lr']:.1e}")
    if it == max_iters: break
    x, y = get_batch(train_data)
    _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
print("done.")

iter    0 | val loss 10.836 | lr 1.0e-05


iter  150 | val loss 5.654 | lr 5.6e-04


iter  300 | val loss 5.230 | lr 3.8e-04


iter  450 | val loss 5.126 | lr 1.6e-04


iter  600 | val loss 5.040 | lr 6.0e-05
done.


## 5. Generate

600 iters on 1 MB of Shakespeare won't be fluent — but you'll see real words and Shakespeare-ish rhythm emerge from pure noise. That's the GPT-3 architecture learning, on your Mac.

In [11]:
model.eval()
start = torch.tensor([enc.encode_ordinary("\n")], device=device)
out = model.generate(start, n=200, temperature=0.8)
print(enc.decode(out[0].tolist()))


Second Citizen:
Well, sir, good.



How we have no, my lord.



What, worthy.
Y:
My lord! I had you shall be so.


GLOUCESCAMILLO:
No, you to many Romeo, sir,
MOPY:
The child.

SICINIUS:
May make thee, for the poor! O, you have I have not know it
A:
GLOUCESTER: but I have not,
I know their blood, I do you,
And he comes the p'd sir.


LADYORK:
What shall be not
IUS:
Sir,, and a king, 'tis a very sovereign!
The point of a lady:
And yet well still.
RICHARD II: what, thou hast thou know you are with death.






## 6. The real thing — scale the 4 dials

Going from this to GPT-3 is **not** a new idea — it's the *same code* with four dials turned up:

1. **Tokenizer** — already BPE (50k pieces). ✓
2. **Width** `n_embd` — 192 here → 768 (GPT-3 Small) → 12288 (175B).
3. **Depth** `n_layer` — 4 here → 12 → 96.
4. **Data** — 1 MB Shakespeare → hundreds of GB of the internet.

### Train the real one on your Mac
The `course/minigpt/` folder has this exact model, sized for an overnight run on **TinyStories** (a dataset engineered so a ~30M model writes *coherent stories*):

```bash
cd course/minigpt
PY="../../.venv/bin/python"
$PY data.py tinystories --mb 200
$PY train.py --preset tinystories --max_iters 12000
$PY sample.py --ckpt ckpt_tinystories.pt --prompt "Once upon a time"
```

**Reality check for a fanless MacBook Air M4 (16 GB):** memory isn't the limit — *heat* is. Train overnight, keep it cool. The true 125M `gpt3-small` preset is in `model.py` for reference, but that one wants a cloud GPU.

### Exercises
1. Bump `n_embd` to 384 and `n_layer` to 6, retrain — does val loss drop?
2. Try `temperature=0.3` vs `1.2` in `generate` — safe vs wild.
3. Swap Tiny Shakespeare for your own `.txt` and watch it learn that style.